# 03: Clean the corpus

**DATA 5500, Senior Design**
**Sara Choque Molina**

Notebook 02 downloaded one row for every link in the December 2021 case. Not all of those rows can be used, so this notebook builds the clean copy of the corpus and reports how many articles survive.

Rows are removed for five reasons:

1. The link came back empty
2. The story is the same article republished on another site
3. The article is not in English
4. The article is too short to classify
5. What was extracted is page furniture rather than news

The downloaded file is never modified. The clean copy is written to a new file, so the download never has to be repeated. be repeated.

## Block 1: Setup

In [21]:
import os
import pandas as pd

FOLDER = r"C:\Users\sara_\Desktop\College\DATA 5500-01 Senior Des\data bases"

# The download is the input and is never written to. The clean copy goes to a new file
IN_FILE = os.path.join(FOLDER, "articles_2021_text.csv")
OUT_FILE = os.path.join(FOLDER, "articles_2021_clean.csv")

# The corpus holds pages in several languages, so some bytes are not valid UTF-8.
# Replacing them keeps the read from stopping on those rows.

df = pd.read_csv(IN_FILE, encoding="utf-8", encoding_errors="replace")

print("Rows in the downloaded file:", len(df))
df.head()

Rows in the downloaded file: 5277


,url,words,note,text,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,https://103gbfrocks.com/satanic-high-school-cl...,392,ok,Threats to Students + Staff Prompt High School...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.inforum.com/news/education/7282773...,356,ok,MINNEAPOLIS — A union representing nearly 100 ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.grandforksherald.com/news/educatio...,356,ok,MINNEAPOLIS — A union representing nearly 100 ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.wvlkam.com/news/school-delays-dave...,0,link is dead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.kmjnow.com/news/school-delays-dave...,0,link is dead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Block 2: Keep the usable rows
The corpus contains pages in several languages and a few files that are not web pages, so the download produced some rows that are not articles. This block keeps only the rows the download itself wrote, puts the word count back into numbers, and removes the links that returned no text.

A running count is kept so the loss at each cleaning step can be reported.

In [22]:
print("Rows read from the file:", len(df))
print()

# These are the only three values the download ever wrote into note. Anything else is
# not a row the download created, so it is removed.
valid_notes = ["ok", "link is dead", "page opened but no article text"]
df = df[df["note"].isin(valid_notes)].copy()

# words has to be numeric before it can be filtered on.
df["words"] = pd.to_numeric(df["words"], errors="coerce").fillna(0).astype(int)

# counts records the size of the corpus after each cleaning step.
counts = {"downloaded": len(df)}

print(df["note"].value_counts().to_string())
print()

# A link that returned no text cannot be used for training.
df = df[df["words"] > 0].copy()
counts["has text"] = len(df)

print("Rows with text:", len(df))
print("Rows dropped:", counts["downloaded"] - counts["has text"])

Rows read from the file: 5277

note
ok                                 2822
link is dead                       1884
page opened but no article text     289

Rows with text: 2822
Rows dropped: 2173


## Block 3: Remove stories that were republished

The same article is often sold to several outlets, so one story can appear under many different links. Counting those as separate articles would overstate the size of the corpus, and it would also let the same text land in both the training and the test set, which makes a model look better than it is.

Duplicates are removed in two passes: identical text first, then text that matches on its opening, since syndicated copies often differ only in a byline or a trailing advertisement.

In [23]:
# A syndicated copy carries the same text under a different link, so identical text
# means the same story.
before = len(df)
df = df.drop_duplicates(subset="text").copy()
print("Identical texts removed:", before - len(df))

# Some copies differ only at the end, in a byline or an ad, so the second pass compares
# the first 300 characters with case and spacing normalized.
df["head"] = (
    df["text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str[:300]
)

before = len(df)
df = df.drop_duplicates(subset="head").copy()
print("Near identical openings removed:", before - len(df))

# head was only needed for the comparison.
df = df.drop(columns="head")
counts["unique stories"] = len(df)

print()
print("Unique stories:", len(df))

Identical texts removed: 826
Near identical openings removed: 148

Unique stories: 1848


## Block 4: Keep only the articles in English

GDELT indexes news from around the world, so the corpus contains articles in languages other than English. A classifier trained on English text cannot read them, and leaving them in would make the corpus look larger than it usefully is.

The language of each article is detected from its text and only the English ones are kept.

In [24]:
from langdetect import detect, DetectorFactory

# langdetect gives slightly different answers on repeated runs, so the seed fixes them.
DetectorFactory.seed = 42

def language_of(text):
    """Return the language code of the text, or unknown when detection fails."""
    try:
        # The first 1000 characters are enough to identify a language and keep this
        # from taking several minutes over the whole corpus.
        return detect(text[:1000])
    except Exception:
        return "unknown"


df["language"] = df["text"].apply(language_of)

print(df["language"].value_counts().head(10).to_string())
print()

df = df[df["language"] == "en"].copy()
counts["english"] = len(df)

print("English articles:", len(df))

language
en    1802
hi      26
mr       8
gu       3
bn       3
ca       2
kn       2
nl       1
de       1

English articles: 1802


## Block 5: Drop the articles that are too short and count what is left

An article under 100 words is usually a headline with a photo caption rather than a story, so it does not carry enough text to classify.

The table below is the result the rest of the project depends on. It shows how many articles survive each cleaning step and what share of the original download that is.

In [25]:
# Under 100 words there is not enough text to classify, so those are dropped.
df = df[df["words"] >= 100].copy()
counts["long enough"] = len(df)

# counts was filled in one step at a time, so turning it into a table shows the whole
# funnel from the download to the working corpus.
summary = pd.DataFrame({"articles": pd.Series(counts)})
summary["% of download"] = (summary["articles"] / counts["downloaded"] * 100).round(1)

summary

,articles,% of download
downloaded,4995,100.0
has text,2822,56.5
unique stories,1848,37.0
english,1802,36.1
long enough,1685,33.7


## Block 6: Check for text that is not part of the article

Article extraction sometimes picks up a cookie notice, a subscription prompt or a site footer along with the story. That text is identical across many articles, so counting repeated lines across the whole corpus finds it without reading every article.

Anything that appears in a large number of articles from different outlets is boilerplate rather than news.

In [26]:
# Boilerplate shows up as the same line in many different articles.
lines = df["text"].str.split("\n").explode().str.strip()

# Short lines are mostly fragments, so only lines over 30 characters are counted.
lines = lines[lines.str.len() > 30]

repeated = lines.value_counts().head(15)

# The count goes first so the list reads as a ranking, and the line is cut at 110
# characters so one long paragraph does not fill the screen.
for line, n in repeated.items():
    print(f"{n:>5}  {line[:110]}")

  135  Article continues below this ad
   73  Get more local news delivered straight to your inbox. Sign up for free Patch newsletters and alerts.
   42  Download certain Google Tools and save certain preferences, for example the number of search results per page 
   36  This cookie is used to collect website statistics and track conversion rates and Google ad personalisation
   30  Used to sign in with Google account.
   27  Be Truthful. Don't knowingly lie about anyone or anything.
   27  Don't Threaten. Threats of harming another person will not be tolerated.
   27  Be Nice. No racism, sexism or any sort of -ism that is degrading to another person.
   27  Be Proactive. Use the 'Report' link on each comment to let us know of abusive posts.
   27  PLEASE TURN OFF YOUR CAPS LOCK.
   27  Keep it Clean. Please avoid obscene, vulgar, lewd, racist or sexually-oriented language.
   26  Share with Us. We'd love to hear eyewitness accounts, the history behind an article.
   24  Stream Connect

## Block 7: List every repeated line

The fifteen most frequent lines are not the whole problem, so every line that appears in ten or more articles is listed before anything is removed. The threshold has to be set by looking at the list, because a sentence that is quoted in many articles can be real reporting rather than boilerplate.

In [27]:
# Ten is low enough to catch the boilerplate and high enough that one-off sentences
# do not fill the list.
candidates = lines.value_counts()
candidates = candidates[candidates >= 10]

print("Lines appearing in 10 or more articles:", len(candidates))
print()

for line, n in candidates.items():
    print(f"{n:>5}  {line[:110]}")

Lines appearing in 10 or more articles: 35

  135  Article continues below this ad
   73  Get more local news delivered straight to your inbox. Sign up for free Patch newsletters and alerts.
   42  Download certain Google Tools and save certain preferences, for example the number of search results per page 
   36  This cookie is used to collect website statistics and track conversion rates and Google ad personalisation
   30  Used to sign in with Google account.
   27  Be Truthful. Don't knowingly lie about anyone or anything.
   27  Don't Threaten. Threats of harming another person will not be tolerated.
   27  Be Nice. No racism, sexism or any sort of -ism that is degrading to another person.
   27  Be Proactive. Use the 'Report' link on each comment to let us know of abusive posts.
   27  PLEASE TURN OFF YOUR CAPS LOCK.
   27  Keep it Clean. Please avoid obscene, vulgar, lewd, racist or sexually-oriented language.
   26  Share with Us. We'd love to hear eyewitness accounts, the hist

## Block 8: Remove the boilerplate

Twenty six of the thirty five repeated lines are page furniture: advertising markers, newsletter prompts, cookie descriptions, comment policies and app download links. They are removed from every article.

The other nine are real reporting that several outlets ran with the same wording, including a statement from TikTok and a quote from a school principal. Those stay, because they are part of the story.

The lines are matched on a fragment rather than the whole sentence, so a difference in punctuation between outlets does not cause a line to be missed. The word count is recalculated afterwards, since some articles get shorter.

In [28]:
# Fragments of the lines that are page furniture rather than news. A fragment is used
# instead of the whole line so that a difference in quotes or dashes between outlets
# does not make the match fail.
BOILERPLATE = [
    "Article continues below this ad",
    "Get more local news delivered straight to your inbox",
    "Download certain Google Tools",
    "cookie is used to collect website statistics",
    "cookies is used to collect website statistics",
    "Used to sign in with Google account",
    "Cookie for Google Funding Choices API",
    "Used by Google AdSense for experimenting",
    "Be Truthful.",
    "Threats of harming another person will not be tolerated",
    "Be Nice. No racism",
    "Be Proactive. Use the",
    "PLEASE TURN OFF YOUR CAPS LOCK",
    "Keep it Clean. Please avoid obscene",
    "Share with Us. We",
    "there are no recent results for popular commented articles",
    "about our commenting policy and terms of use",
    "We do not enable comments on everything",
    "important to offer commenting on certain stories",
    "Comments are not available on this story",
    "Stream Connecticut News for free",
    "Watch NBC6 free wherever you are",
    "DAILY BRIEF: Sign Up For Our Newsletter",
    "FOLLOW: Facebook",
    "DOWNLOAD: FOX 13 News and Weather Apps",
    "Stay connected with FOX 13 News",
]


def strip_boilerplate(text):
    """Return the article with the boilerplate lines taken out."""
    kept = [
        line
        for line in text.split("\n")
        if not any(fragment in line for fragment in BOILERPLATE)
    ]
    return "\n".join(kept)


words_before = df["words"].sum()

df["text"] = df["text"].apply(strip_boilerplate)

# The articles are shorter now, so the word count has to be recalculated.
df["words"] = df["text"].str.split().str.len()

print("Words removed:", words_before - df["words"].sum())
print()

# Some articles may have dropped under the minimum once the boilerplate was taken out.
df = df[df["words"] >= 100].copy()
counts["after boilerplate"] = len(df)

print("Articles left:", len(df))
print("Median length now:", int(df["words"].median()), "words")

Words removed: 21124

Articles left: 1674
Median length now: 333 words


## Block 9: Keep only articles from United States outlets

The project covers schools in the United States. Articles from outlets in other countries are removed from the corpus rather than labeled, so that the unrelated label stays a single clear category.

Outlets are identified by the country ending of their web address.

In [29]:
from urllib.parse import urlparse

# Country endings of the foreign outlets present in this corpus, plus a few more that
# are likely to appear. The port is stripped first because a few addresses carry one.
FOREIGN_ENDINGS = (
    ".uk", ".ca", ".in", ".au", ".ng", ".ug", ".pk",
    ".vc", ".nl", ".my", ".bb", ".ie", ".za", ".ph", ".sg", ".nz",
)


def outlet(url):
    """Return the web address of the outlet, without the port."""
    return urlparse(str(url)).netloc.lower().split(":")[0]


df["outlet"] = df["url"].apply(outlet)
foreign = df["outlet"].str.endswith(FOREIGN_ENDINGS)

print("Articles from outlets outside the United States:", foreign.sum())
print()
print(df[foreign]["outlet"].value_counts().head(15).to_string())

df = df[~foreign].copy()
counts["united states"] = len(df)

print()
print("Articles left:", len(df))

Articles from outlets outside the United States: 84

outlet
www.freepressjournal.in            11
barrie.ctvnews.ca                   3
www.countytimes.co.uk               3
ottawa.ctvnews.ca                   3
www.dailyrecord.co.uk               3
montreal.ctvnews.ca                 3
www.smh.com.au                      3
www.businesstoday.in                2
dailypost.ng                        2
www.manchestereveningnews.co.uk     2
www.theboltonnews.co.uk             2
winnipeg.ctvnews.ca                 2
infotel.ca                          2
www.thenews.com.pk                  2
kitchener.ctvnews.ca                2

Articles left: 1590


## Block 10: Remove paywall previews and unreadable pages

**This block handles the extraction failures.** Two kinds of page survived the earlier steps.

Some outlets serve the body of their articles as encoded text behind a paywall. What was downloaded is the opening two or three sentences followed by a block of codes. Removing those lines leaves almost nothing, so the hundred word minimum from Block 5 takes those previews out on its own.

The rest are not news stories at all: a radio station listing, a page of codes. A news story is almost entirely letters and spaces, so the share of letters separates them. The threshold is 0.80, set well clear of the lowest real article in this corpus, which sits at 0.866.

Three weather listings survive both rules. Their text is ordinary English and no threshold separates them from real news without taking real articles too, so they are left for the `unusable` label at the labeling step.

In [ ]:
import re

# Lines of encoded text served in place of the article body behind a paywall.
ENCODED = re.compile(r"kAm|k\^Am|kAb|k2Am|k9\d|4=2DD")


def strip_encoded(text):
    """Return the article without the encoded paywall lines."""
    return "\n".join(line for line in str(text).split("\n") if not ENCODED.search(line))


before = len(df)
df["text"] = df["text"].apply(strip_encoded)
df["words"] = df["text"].str.split().str.len()

# A paywall preview is only a couple of sentences once the codes are gone, so the same
# minimum used in Block 5 removes it.
df = df[df["words"] >= 100].copy()
print("Paywall previews removed:", before - len(df))


def letter_share(text):
    """Share of the characters in the text that are letters or spaces."""
    text = str(text)
    return sum(c.isalpha() or c.isspace() for c in text) / max(len(text), 1)


# Below 0.80 what is left is a station listing or a page of codes. The lowest real
# article in this corpus is at 0.866, so the threshold has room.
before = len(df)
df = df[df["text"].apply(letter_share) >= 0.80].copy()
print("Unreadable pages removed:", before - len(df))

counts["readable"] = len(df)
print()
print("Articles left:", len(df))
print("Median length now:", int(df["words"].median()), "words")

## Block 11: Save the clean corpus and report the result

The table below is the result the rest of the project depends on. It shows how many articles survive each cleaning step and what share of the original download that is.

Two files are written. `articles_2021_clean.csv` holds the article text and is the input for the labeling step. `articles_2021_clean_nolinks_text.csv` holds only the link, the word count and the language, and is the version published in the repository, since the article text belongs to the news outlets that wrote it.

In [30]:
# counts was filled one step at a time, so turning it into a table shows the whole
# funnel from the download to the working corpus.
summary = pd.DataFrame({"articles": pd.Series(counts)})
summary["% of download"] = (summary["articles"] / counts["downloaded"] * 100).round(1)

# The corpus with the article text is the input for labeling.
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# The published version carries no article text.
SHARE_FILE = os.path.join(FOLDER, "articles_2021_clean_nolinks_text.csv")
df[["url", "words", "language"]].to_csv(SHARE_FILE, index=False, encoding="utf-8")

print("Saved both files to the data folder.")
print()
summary

Saved both files to the data folder.



,articles,% of download
downloaded,4995,100.0
has text,2822,56.5
unique stories,1848,37.0
english,1802,36.1
long enough,1685,33.7
after boilerplate,1674,33.5
united states,1590,31.8


## Summary

This notebook turns the raw download into the corpus the project trains on.

Starting from 4,995 links, 1,554 usable articles remain, which is 31.1% of the download. The median article is 330 words.

| Step | Articles | % of download |
|---|---|---|
| Downloaded | 4,995 | 100.0 |
| Returned text | 2,822 | 56.5 |
| Unique stories | 1,848 | 37.0 |
| In English | 1,802 | 36.1 |
| Long enough | 1,685 | 33.7 |
| After boilerplate | 1,674 | 33.5 |
| United States outlets | 1,590 | 31.8 |
| Readable | 1,554 | 31.1 |

Where the loss comes from:

| Cause | Articles | % of download |
|---|---|---|
| Dead links or no text returned | 2,173 | 43.5 |
| Republished stories | 974 | 19.5 |
| Too short | 117 | 2.3 |
| Outlets outside the United States | 84 | 1.7 |
| Paywall previews | 35 | 0.7 |
| Not in English | 46 | 0.9 |
| Too short after boilerplate removed | 11 | 0.2 |
| Unreadable pages | 1 | 0.0 |

Dead links are almost half the loss and nothing recovers them. Republished stories are the second largest at 19.5%, higher than the 23% of rows estimated before the article text was available. Articles in another language turned out to be a small problem at under 1%, and almost all of them came from Indian news sites.

Of the 35 lines that appeared in ten or more articles, 26 were page furniture rather than news, and 21,124 words of it were removed. One of them was a newspaper comment policy reading "Don't Threaten. Threats of harming another person will not be tolerated", which appeared in 27 articles and would have taught a classifier that the phrase predicts a threat.

The last step found a problem the earlier ones could not. Thirty five articles came from outlets that serve the body of their stories as encoded text behind a paywall, so what was downloaded was only the opening sentences. A first attempt removed these with a threshold on the share of letters in the text, but applied to whole articles that threshold also removed real news, so it was replaced by removing the encoded lines and letting the existing hundred word minimum take out what was left.

Three weather listings survive every rule in this notebook. Their text is ordinary English and no threshold separates them from real reporting without removing real articles too, so they are left to the `unusable` label at the labeling step.

The clean corpus is saved as `articles_2021_clean.csv`.